### Import Libraries

In [4]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

StatementMeta(, 6ffbfb45-4813-4ffa-bb66-8c41ce52c6ad, 6, Finished, Available, Finished, False)

### Exploar Customer Data 

In [5]:
## Read Customer Data
customers = spark.table("customers")
display(customers.limit(5))

StatementMeta(, 6ffbfb45-4813-4ffa-bb66-8c41ce52c6ad, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 65114dd7-b017-46ad-84f2-bbf978931f54)

### Clean Customers Data 

In [6]:
customers_clean= (
    customers
    .withColumn("email",lower(trim(col("EMAIL"))))
    .withColumn("name",initcap(trim(col("name"))))
    .withColumn("gender",
                when(lower(col("gender")).isin("f","female"),"Female")
               .when(lower(col("gender")).isin("m","male"),"Male").otherwise("other"))
    .withColumn("dob",to_date(regexp_replace(col("dob"),"/","-")))
    .withColumn("location",initcap(col("location")))
    .dropDuplicates(["customer_id"])
    .dropna(subset=["customer_id","email"])
)
display(customers_clean.limit(5))
customers_clean.write.format("delta").mode("overwrite").saveAsTable("silver_customers")

StatementMeta(, 6ffbfb45-4813-4ffa-bb66-8c41ce52c6ad, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f8cea157-c67d-46dd-9395-2ca72be65a78)

### Clean Orders Data

In [7]:
orders = spark.table("orders")
display(orders.limit(5))

StatementMeta(, 6ffbfb45-4813-4ffa-bb66-8c41ce52c6ad, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8b43f85d-39c2-462b-8376-bbe03f589a3a)

In [8]:
orders_clean=(
    orders
    .withColumn("order_date",
                when(col("order_date").rlike("^\d{4}/\d{2}/\d{2}$"), to_date(col("order_date"), "yyyy/MM/dd"))
                .when(col("order_date").rlike("^\d{2}-\d{2}-\d{4}$"), to_date(col("order_date"), "dd-MM-yyyy"))
                .when(col("order_date").rlike("^\d{8}$"), to_date(col("order_date"), "yyyyMMdd"))
                .otherwise(to_date(col("order_date"), "yyyy-MM-dd")))
                .withColumn("amount",col("amount").cast(DoubleType()))
                .withColumn("amount",when(col("amount") < 0 ,None).otherwise(col("amount")))
                .withColumn("status",initcap(col("status")))
                .dropna(subset=["customer_id","order_date"])
                .dropDuplicates(["order_id"])
)
display(orders_clean.limit(5))
orders_clean.write.format("delta").mode("overwrite").saveAsTable("silver_orders")

StatementMeta(, 6ffbfb45-4813-4ffa-bb66-8c41ce52c6ad, 10, Finished, Available, Finished, False)

<>:4: SyntaxWarning: invalid escape sequence '\d'
<>:5: SyntaxWarning: invalid escape sequence '\d'
<>:6: SyntaxWarning: invalid escape sequence '\d'
<>:4: SyntaxWarning: invalid escape sequence '\d'
<>:5: SyntaxWarning: invalid escape sequence '\d'
<>:6: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_6002/2013498930.py:4: SyntaxWarning: invalid escape sequence '\d'
  when(col("order_date").rlike("^\d{4}/\d{2}/\d{2}$"), to_date(col("order_date"), "yyyy/MM/dd"))
/tmp/ipykernel_6002/2013498930.py:5: SyntaxWarning: invalid escape sequence '\d'
  .when(col("order_date").rlike("^\d{2}-\d{2}-\d{4}$"), to_date(col("order_date"), "dd-MM-yyyy"))
/tmp/ipykernel_6002/2013498930.py:6: SyntaxWarning: invalid escape sequence '\d'
  .when(col("order_date").rlike("^\d{8}$"), to_date(col("order_date"), "yyyyMMdd"))


SynapseWidget(Synapse.DataFrame, 6c8828a0-f4aa-4002-ae92-da410a39eee6)

### Clean Orders Data

In [11]:
%%sql
select * from payments limit 5

StatementMeta(, 6ffbfb45-4813-4ffa-bb66-8c41ce52c6ad, 11, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 6 fields>

In [17]:
payments= spark.table("payments")
payments_clean=(
    payments
    .withColumn("payment_date",to_date(regexp_replace(col("payment_date"),"/","-")))
    .withColumn("payment_method",initcap(col("payment_method")))
    .replace({"creaditcard":"Creadit Card"},subset=["payment_method"])
    .withColumn("payment_status",initcap(col("payment_status")))
    .withColumn("amount",col("amount").cast(DoubleType()))
    .withColumn("amount",when(col("amount") < 0 ,None).otherwise(col("amount")))
    .dropna(subset=["customer_id","payment_date","amount"])
)
display(payments_clean)
payments_clean.write.format("delta").mode("overwrite").saveAsTable("Silver_Payments")

StatementMeta(, 6ffbfb45-4813-4ffa-bb66-8c41ce52c6ad, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 36d86a3b-06bf-4de7-87cb-e618fe1c2bb3)

### Clean support Data

In [18]:
%%sql
select * from support limit 5 

StatementMeta(, 6ffbfb45-4813-4ffa-bb66-8c41ce52c6ad, 18, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 5 fields>

In [21]:
support = spark.table("support")
clean_support=(
    support
    .withColumn("ticket_date",to_date(regexp_replace(col("ticket_date"),"/","-")))
    .withColumn("issue_type",initcap(trim(col("issue_type"))))
    .withColumn("resolution_status",initcap(trim(col("resolution_status"))))
    .replace({"NA":None,"":None},subset=["issue_type","resolution_status"])
    .dropDuplicates(["ticket_id"])
    .dropna(subset=["customer_id","ticket_date"])
)
display(clean_support)
clean_support.write.format("delta").mode("overwrite").saveAsTable("Silver_Support")

StatementMeta(, 6ffbfb45-4813-4ffa-bb66-8c41ce52c6ad, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a7e7ccbd-58db-4531-a881-e05b41f9facd)

### Clean Web Data

In [22]:
%%sql
select * from web limit 5

StatementMeta(, 6ffbfb45-4813-4ffa-bb66-8c41ce52c6ad, 22, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 5 fields>

In [27]:
web=spark.table("web")
clean_web=(
    web
    .withColumn("session_time",to_date(regexp_replace(col("session_time"),"/","-")))
    .withColumn("page_viewed",lower(col("page_viewed")))
    .withColumn("device_type",initcap(col("device_type")))
    .dropDuplicates(["session_id"])
    .dropna(subset=["customer_id","session_id","page_viewed"])
)
display(clean_web)
clean_web.write.format("delta").mode("overwrite").saveAsTable("Silver_Web")

StatementMeta(, 6ffbfb45-4813-4ffa-bb66-8c41ce52c6ad, 27, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, edc36028-b7aa-4298-838d-496c4bcfe6cc)